In [1]:
ACCESS_TOKEN = "54D1CAB33916ADAE005243953A000046898E36F2E8A0E15AEDC83BEFC76F473C"

In [2]:
import requests
import time
import pandas as pd
from datetime import datetime, timedelta

# --- CẤU HÌNH ---
BASE_URL = "https://apis.haravan.com/com/orders.json"
OUTPUT_FILE = r"N:\160 - IT\020 - DATA\Data\Haravan_Order.parquet"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}

In [3]:
def fetch_haravan_orders_6months():
    page = 1
    all_orders = []
    
    # 1. Tự động tính mốc Max: 23:59:59 ngày hôm qua
    yesterday = datetime.now() - timedelta(days=1)
    max_date_str = yesterday.strftime('%Y-%m-%dT23:59:59.000Z')
    
    # 2. Tự động tính mốc Min: Lùi lại 180 ngày (6 tháng) từ hôm nay
    six_months_ago = datetime.now() - timedelta(days=180)
    min_date_str = six_months_ago.strftime('%Y-%m-%dT00:00:00.000Z')

    print(f"BẮT ĐẦU KÉO DỮ LIỆU 6 THÁNG GẦN NHẤT")
    print(f"Khoảng thời gian: {min_date_str} -> {max_date_str}")
    print("-" * 40)
    
    while True:
        params = {
            "status": "any",
            "created_at_min": min_date_str,
            "created_at_max": max_date_str,
            "order": "created_at asc",
            "limit": 50,
            "page": page
        }
        
        try:
            response = requests.get(BASE_URL, headers=headers, params=params)
            
            if response.status_code == 429:
                retry_after = float(response.headers.get("Retry-After", 2.0))
                print(f"[CẢNH BÁO] Chạm ngưỡng API. Ngủ {retry_after} giây...")
                time.sleep(retry_after)
                continue
                
            elif response.status_code == 200:
                data = response.json()
                orders = data.get("orders", [])
                
                if not orders:
                    print(f"\n-> Đã quét hết dữ liệu 6 tháng. Hoàn tất!")
                    break
                    
                all_orders.extend(orders)
                print(f"Đã lấy trang {page}: {len(orders)} đơn hàng.")
                
                time.sleep(0.5) 
                page += 1
            else:
                print(f"[LỖI {response.status_code}] {response.text}")
                break
        except Exception as e:
            print(f"❌ Lỗi kết nối: {e}")
            break
            
    return all_orders

In [4]:
def export_haravan_order_lifecycle(orders_data, output_file):
    order_list = []
    print("\nĐang trích xuất dữ liệu trạng thái đơn hàng (Lifecycle)...")
    
    for order in orders_data:
        # 1. Trích xuất UTM từ note_attributes
        note_attrs = order.get('note_attributes', [])
        utm_source = next((attr.get('value') for attr in note_attrs if 'source' in attr.get('name', '').lower()), None)
        utm_campaign = next((attr.get('value') for attr in note_attrs if 'campaign' in attr.get('name', '').lower()), None)

        # 2. Thu thập các mốc thời gian và trạng thái
        row = {
            "Order Number": order.get('name') or order.get('order_number'),
            "Prev Order Number": order.get('prev_order_number'),
            "Ref Order Number": order.get('ref_order_number'),
            
            "Created At": order.get('created_at'),
            "Closed At": order.get('closed_at'),
            "Cancelled At": order.get('cancelled_at'),
            "Updated At": order.get('updated_at'),
            "Confirmed At": order.get('confirmed_at'),
            "Ref Order Date": order.get('ref_order_date'),
            "Prev Order Date": order.get('prev_order_date'),
            
            "Financial Status": order.get('financial_status'),
            "Fulfillment Status": order.get('fulfillment_status'),
            "Order Processing Status": order.get('order_processing_status'),
            "Cancel Reason": order.get('cancel_reason'),
            
            "Gateway": order.get('gateway'),
            "Source": order.get('source_name'),
            "UTM Source": utm_source,
            "UTM Campaign": utm_campaign,
            "Note": order.get('note'),
            "Total Price": float(order.get('total_price') or 0)
        }
        order_list.append(row)

    if order_list:
        df = pd.DataFrame(order_list)
            
        # Sắp xếp theo ngày tạo mới nhất và ghi đè ra file
        df = df.sort_values(by="Created At", ascending=False)
        df.to_parquet(output_file, engine='pyarrow', compression='snappy', index=False)
        print(f"✅ Đã xuất {len(df)} đơn hàng (6 tháng qua) tại: {output_file}")
    else:
        print("⚠️ Không có dữ liệu để xuất.")

if __name__ == "__main__":
    # Bước 1: Kéo dữ liệu
    raw_data = fetch_haravan_orders_6months()
    
    # Bước 2: Xử lý và Xuất file
    if raw_data:
        export_haravan_order_lifecycle(raw_data, OUTPUT_FILE)

BẮT ĐẦU KÉO DỮ LIỆU 6 THÁNG GẦN NHẤT
Khoảng thời gian: 2025-10-01T00:00:00.000Z -> 2026-03-29T23:59:59.000Z
----------------------------------------


Đã lấy trang 1: 50 đơn hàng.


Đã lấy trang 2: 50 đơn hàng.


Đã lấy trang 3: 50 đơn hàng.


Đã lấy trang 4: 50 đơn hàng.


...

Đã lấy trang 305: 50 đơn hàng.


Đã lấy trang 306: 50 đơn hàng.


Đã lấy trang 307: 21 đơn hàng.



-> Đã quét hết dữ liệu 6 tháng. Hoàn tất!

Đang trích xuất dữ liệu trạng thái đơn hàng (Lifecycle)...


✅ Đã xuất 15321 đơn hàng (6 tháng qua) tại: N:\160 - IT\020 - DATA\Data\Haravan_Order.parquet


In [5]:
# import pandas as pd
# import os

# def convert_xlsx_to_parquet(input_xlsx, output_parquet):
#     print(f"--- ĐANG CHUYỂN ĐỔI: {os.path.basename(input_xlsx)} ---")
    
#     try:
#         # 1. Đọc dữ liệu từ Excel
#         # engine='openpyxl' là chuẩn nhất cho file .xlsx
#         df = pd.read_excel(input_xlsx, engine='openpyxl')
        
#         # 2. Kiểm tra và chuẩn hóa kiểu dữ liệu (Rất quan trọng cho Parquet)
#         # Chuyển các cột ngày tháng về đúng định dạng datetime
            
#         # Đảm bảo các cột giá trị là số
#         if 'Total Price' in df.columns:
#             df['Total Price'] = pd.to_numeric(df['Total Price'], errors='coerce').fillna(0)

#         # 3. Ghi đè ra file Parquet
#         # compression='snappy' là sự cân bằng tốt nhất giữa tốc độ và dung lượng
#         df.to_parquet(output_parquet, engine='pyarrow', compression='snappy', index=False)
        
#         print(f"✅ Thành công! File Parquet đã được tạo: {output_parquet}")
#         print(f"Dung lượng file mới: {os.path.getsize(output_parquet) / 1024:.2f} KB")

#     except Exception as e:
#         print(f"❌ Lỗi khi chuyển đổi: {e}")

# # --- THỰC THI ---
# file_excel = r"D:\Work\pyWork\Haravan_Order_Lifecycle.xlsx"
# file_parquet = r"D:\Work\pyWork\Haravan_Order.parquet"

# convert_xlsx_to_parquet(file_excel, file_parquet)